The API https://www.nohrsc.noaa.gov/nsa/discussions_text/National/snowdepth/202601/snowdepth_2026011806_m.txt contains the extension _2026011806, or 
_YYYYMMDD06

In [1]:
def download_snow_data(start_date, end_date, destination="snow_data_extraction/data"):
    from datetime import datetime, timedelta
    from pathlib import Path
    from urllib.error import HTTPError, URLError
    from urllib.request import urlopen
    
    if isinstance(start_date, str):
        start_date = datetime.strptime(start_date, "%Y-%m-%d")
    if isinstance(end_date, str):
        end_date = datetime.strptime(end_date, "%Y-%m-%d")
    
    start_date = start_date.date() if hasattr(start_date, "date") else start_date
    end_date = end_date.date() if hasattr(end_date, "date") else end_date
    
    if end_date < start_date:
        raise ValueError("end_date must be on or after start_date")
    
    destination_path = Path(destination)
    destination_path.mkdir(parents=True, exist_ok=True)
    
    current_date = start_date
    saved_files = []
    
    while current_date <= end_date:
        yyyymm = current_date.strftime("%Y%m")
        stamp = current_date.strftime("%Y%m%d") + "06"
        filename = f"snowdepth_{stamp}_m.txt"
        source_url = (
            "https://www.nohrsc.noaa.gov/nsa/discussions_text/National/snowdepth/"
            f"{yyyymm}/{filename}"
        )
        try:
            with urlopen(source_url, timeout=30) as response:
                payload = response.read().decode("utf-8")
        except HTTPError as exc:
            raise RuntimeError(f"Failed to download {source_url}: HTTP {exc.code}") from exc
        except URLError as exc:
            raise RuntimeError(f"Failed to download {source_url}: {exc.reason}") from exc
    
        output_path = destination_path / filename
        output_path.write_text(payload, encoding="utf-8")
        saved_files.append(str(output_path))
        current_date += timedelta(days=1)
    
    return saved_files

In [ ]:
#test function on first 5 days of year
download_snow_data("2026-02-23", "2026-03-01", destination="../client/public/snow_data")

['../client/public/snowdepth_2026022306_m.txt',
 '../client/public/snowdepth_2026022406_m.txt',
 '../client/public/snowdepth_2026022506_m.txt',
 '../client/public/snowdepth_2026022606_m.txt',
 '../client/public/snowdepth_2026022706_m.txt',
 '../client/public/snowdepth_2026022806_m.txt',
 '../client/public/snowdepth_2026030106_m.txt']